# Local-SDFT: Fine-tune a 230M model on Colab

Self-Distillation Fine-Tuning (**SDFT**) is on-policy self-distillation from demos: show the model a few gold pairs, let it answer, then LoRA-train on those self-generated targets — instead of plain SFT on off-policy gold text. (Shenfeld et al. study this for continual learning / less forgetting; here we run the same loop on a laptop-sized model.)

**This notebook is standalone** — no `git clone`, no Local-SDFT package import. Helpers for load / generate / train / judge are inlined below.

| Arm | What it does |
|-----|----------------|
| **ZS** | Zero-shot base model (inference) |
| **ICL** | Few-shot in-context learning (3 demos from alpaca-cleaned) |
| **CoT** | Chain-of-thought cue (`Let's think step by step.`) |
| **gold SFT** | LoRA train on `yahma/alpaca-cleaned` gold `output` |
| **SDFT** | Teacher generate → LoRA on self-distilled targets |

Paper: [Shenfeld et al., 2026 — *Self-Distillation Enables Continual Learning*](https://self-distillation.github.io/SDFT) ([arXiv](https://arxiv.org/abs/2601.19897)).

Base model: [Liquid AI LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M).

**Protocol:** train gold SFT / SDFT on **256** `yahma/alpaca-cleaned` rows; generate on all **805** AlpacaEval 2.0 instructions; pairwise vs `gpt4_turbo` reference with hard-coded **Qwen/Qwen3.5-9B** 4-bit local judge (≈ AE2 protocol, not LC leaderboard). Expect **~1–3+ hours** on Colab T4.


## Setup

- **Colab**: Runtime → GPU (T4 is fine).
- Installs `transformers` / `peft` / `trl` / … and uninstalls stale `torchao` so peft LoRA does not hit a version gate.
- No repo clone. Working directory can be anywhere.


In [ ]:
# peft>=0.19 raises ImportError if torchao is installed but <0.16 (Colab often ships 0.10).
%pip install -q "transformers>=4.54" "trl>=0.19" "peft>=0.15" "datasets>=2.19" "accelerate>=0.33" "pyyaml>=6.0" "bitsandbytes>=0.43" "huggingface_hub>=0.23"
%pip uninstall -y torchao

print("setup ok — local judge: Qwen/Qwen3.5-9B (4-bit); ≈ AE2 protocol, not LC leaderboard")


## Helpers (inlined)

Minimal load / generate / train / judge utilities — no Local-SDFT package.


In [ ]:
import gc
import hashlib
import json
import random
import re
from pathlib import Path
from typing import Any, Sequence

import torch
from datasets import Dataset, load_dataset
from huggingface_hub import hf_hub_download
from peft import LoraConfig as PeftLoraConfig
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

# ---------------------------------------------------------------------------
# Hard-coded constants
# ---------------------------------------------------------------------------
MODEL_NAME = "LiquidAI/LFM2.5-230M"
JUDGE_MODEL = "Qwen/Qwen3.5-9B"

NUM_TRAIN = 256
NUM_EVAL = 805  # full AlpacaEval 2.0
SEED = 0

TEACHER_NUM_SHOTS = 2
TEACHER_BATCH_SIZE = 8
MAX_NEW_TOKENS = 128
MIN_RESPONSE_CHARS = 1

TRAIN_BATCH_SIZE = 16
TRAIN_EPOCHS = 1
TRAIN_LR = 1e-4
TRAIN_MAX_LENGTH = 512
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET = r".*self_attn\.(q|k|v|out)_proj"

ICL_K = 3
COT_LINE = "Let's think step by step."

ALPACA_EVAL_REPO = "tatsu-lab/alpaca_eval"
ALPACA_EVAL_JSON = "alpaca_eval.json"
GPT4_TURBO_REF_JSON = "alpaca_eval_gpt4_baseline.json"

OUT_ROOT = Path("outputs/colab_sdft")
GOLD_PATH = OUT_ROOT / "gold.jsonl"
SDFT_PATH = OUT_ROOT / "sdft.jsonl"
SFT_ADAPTER = OUT_ROOT / "adapter-sft-gold"
SDFT_ADAPTER = OUT_ROOT / "adapter-sdft"
AE_ROOT = OUT_ROOT / "alpacaeval"

REFUSAL_RE = re.compile(r"(?i)\b(i('m| am) sorry|i can('t|not) (assist|help)|as an ai)\b")
_EXPLICIT_A = re.compile(r"output\s*\(\s*a\s*\)", re.IGNORECASE)
_EXPLICIT_B = re.compile(r"output\s*\(\s*b\s*\)", re.IGNORECASE)
_LETTER_A = re.compile(r"(?:^|[^a-z0-9])\(?\s*a\s*\)?(?:[^a-z0-9]|$)", re.IGNORECASE)
_LETTER_B = re.compile(r"(?:^|[^a-z0-9])\(?\s*b\s*\)?(?:[^a-z0-9]|$)", re.IGNORECASE)

_PAIRWISE_USER = """\
Select the output (a) or (b) that best matches the given instruction. Choose \
your preferred output, which can be subjective. Your answer should ONLY \
contain: Output (a) or Output (b). Do not explain.

## Instruction:
{instruction}

## Output (a):
{output_1}

## Output (b):
{output_2}

## Which is best, Output (a) or Output (b)?
"""


def release_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def load_tokenizer(name: str = MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def load_policy(name: str = MODEL_NAME):
    """Load LFM2.5-230M with device_map=auto on CUDA."""
    kwargs: dict[str, Any] = {"dtype": torch.float32}
    if torch.cuda.is_available():
        kwargs["device_map"] = "auto"
        return AutoModelForCausalLM.from_pretrained(name, **kwargs)
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return AutoModelForCausalLM.from_pretrained(name, **kwargs).to("mps")
    return AutoModelForCausalLM.from_pretrained(name, **kwargs)


def to_model_device(batch, model):
    device = next(model.parameters()).device
    if hasattr(batch, "to"):
        return batch.to(device)
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


def load_train_examples(n: int = NUM_TRAIN, seed: int = SEED) -> list[dict]:
    ds = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=seed)
    out = []
    for row in ds.select(range(min(n, len(ds)))):
        parts = []
        for f in ("instruction", "input"):
            v = str(row.get(f) or "").strip()
            if v:
                parts.append(v)
        prompt = "\n\n".join(parts)
        response = str(row.get("output") or "").strip()
        if prompt and response:
            out.append({"prompt": prompt, "response": response})
    return out


def load_eval_examples(n: int = NUM_EVAL) -> list[dict]:
    path = Path(hf_hub_download(ALPACA_EVAL_REPO, ALPACA_EVAL_JSON, repo_type="dataset"))
    rows = json.loads(path.read_text(encoding="utf-8"))
    out = []
    for row in rows:
        if not isinstance(row, dict):
            continue
        prompt = str(row.get("instruction") or "").strip()
        response = str(row.get("output") or "").strip()
        if prompt:
            out.append({"prompt": prompt, "response": response})
    return out[:n]


def load_gpt4_turbo_reference(n: int | None = None) -> list[dict]:
    path = Path(hf_hub_download(ALPACA_EVAL_REPO, GPT4_TURBO_REF_JSON, repo_type="dataset"))
    rows = json.loads(path.read_text(encoding="utf-8"))
    out = []
    for row in rows:
        if not isinstance(row, dict):
            continue
        instruction = str(row.get("instruction") or "").strip()
        output = str(row.get("output") or "").strip()
        if instruction:
            out.append({
                "instruction": instruction,
                "output": output,
                "generator": str(row.get("generator") or "gpt4_turbo"),
            })
    return out if n is None else out[:n]


def sample_fewshots(examples, exclude_idx, num_shots, rng):
    pool = list(range(len(examples)))
    pool.remove(exclude_idx)
    idxs = rng.sample(pool, min(num_shots, len(pool)))
    return [examples[i] for i in idxs]


def build_teacher_messages(fewshots, prompt):
    messages = []
    for shot in fewshots:
        messages.append({"role": "user", "content": shot["prompt"]})
        messages.append({"role": "assistant", "content": shot["response"]})
    messages.append({"role": "user", "content": prompt})
    return messages


def build_eval_messages(instruction, *, few_shots=None, cot=False):
    messages = []
    for shot in few_shots or []:
        messages.append({"role": "user", "content": shot["prompt"]})
        messages.append({"role": "assistant", "content": shot["response"]})
    user = instruction.strip()
    if cot:
        if not user.endswith(COT_LINE):
            user = f"{user}\n\n{COT_LINE}" if user else COT_LINE
    messages.append({"role": "user", "content": user})
    return messages


def pick_icl_demos(train_ex, k=ICL_K, seed=SEED):
    rng = random.Random(seed)
    idxs = rng.sample(range(len(train_ex)), min(k, len(train_ex)))
    return [train_ex[i] for i in sorted(idxs)]


@torch.inference_mode()
def teacher_generate(examples, *, num_shots=TEACHER_NUM_SHOTS, batch_size=TEACHER_BATCH_SIZE,
                     max_new_tokens=MAX_NEW_TOKENS):
    tokenizer = load_tokenizer()
    tokenizer.padding_side = "left"
    model = load_policy()
    model.eval()
    rng = random.Random(SEED)
    results = [None] * len(examples)
    try:
        for start in range(0, len(examples), batch_size):
            batch = examples[start : start + batch_size]
            prompts = [
                tokenizer.apply_chat_template(
                    build_teacher_messages(
                        sample_fewshots(examples, start + i, num_shots, rng),
                        ex["prompt"],
                    ),
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for i, ex in enumerate(batch)
            ]
            enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False)
            enc = to_model_device(enc, model)
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            new = out[:, enc["input_ids"].shape[1] :]
            for i, text in enumerate(tokenizer.batch_decode(new, skip_special_tokens=True)):
                text = text.strip()
                results[start + i] = text if len(text) >= MIN_RESPONSE_CHARS else None
            print(f"  teacher {min(start + batch_size, len(examples))}/{len(examples)}", flush=True)
        return results
    finally:
        del model, tokenizer
        release_gpu()


def train_lora(jsonl_path: Path, output_dir: Path, *, target_field: str):
    rows = [json.loads(l) for l in jsonl_path.read_text().splitlines() if l.strip()]
    ds_rows = []
    for row in rows:
        completion = str(row.get(target_field) or "").strip()
        if target_field == "sdft_response" and not completion:
            completion = str(row.get("response") or "").strip()
        if not completion:
            continue
        ds_rows.append({
            "prompt": [{"role": "user", "content": row["prompt"]}],
            "completion": [{"role": "assistant", "content": completion}],
        })
    ds = Dataset.from_list(ds_rows)
    print(f"training {len(ds)} pairs → {output_dir} (target={target_field})", flush=True)

    tokenizer = load_tokenizer()
    model = load_policy()
    model.config.use_cache = False
    peft_config = PeftLoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET,
        task_type="CAUSAL_LM",
    )
    sft_config = SFTConfig(
        output_dir=str(output_dir),
        num_train_epochs=TRAIN_EPOCHS,
        learning_rate=TRAIN_LR,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=1,
        max_length=TRAIN_MAX_LENGTH,
        warmup_steps=0,
        logging_steps=4,
        save_strategy="no",
        seed=SEED,
        report_to=[],
        completion_only_loss=True,
        use_cpu=not (torch.cuda.is_available() or (hasattr(torch.backends, "mps") and torch.backends.mps.is_available())),
        dataset_num_proc=1,
    )
    trainer = None
    try:
        trainer = SFTTrainer(
            model=model,
            args=sft_config,
            train_dataset=ds,
            peft_config=peft_config,
            processing_class=tokenizer,
        )
        trainer.train()
        trainer.save_model(str(output_dir))
        tokenizer.save_pretrained(str(output_dir))
        print(f"saved adapter → {output_dir}", flush=True)
    finally:
        del trainer, model, tokenizer
        release_gpu()


def parse_pairwise_verdict(text: str) -> int | None:
    if not text or not str(text).strip():
        return None
    raw = str(text).strip()
    first = raw.splitlines()[0].strip()
    for chunk in (first, raw):
        ea, eb = _EXPLICIT_A.search(chunk), _EXPLICIT_B.search(chunk)
        if ea and not eb:
            return 1
        if eb and not ea:
            return 2
        if ea and eb:
            return 1 if ea.start() < eb.start() else 2
    compact = re.sub(r"\s+", " ", first).strip().strip("\"'`.,;:")
    if compact == "m":
        return 1
    if compact == "M":
        return 2
    low = compact.lower()
    if low in {"a", "(a)", "a)", "output a", "output (a)"}:
        return 1
    if low in {"b", "(b)", "b)", "output b", "output (b)"}:
        return 2
    for chunk in (first, raw):
        ma, mb = _LETTER_A.search(chunk), _LETTER_B.search(chunk)
        if ma and not mb:
            return 1
        if mb and not ma:
            return 2
        if ma and mb:
            return 1 if ma.start() < mb.start() else 2
    return None


def _should_swap(instruction: str) -> bool:
    digest = hashlib.md5(instruction.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % 2 == 1


def _winrate_from_preferences(preferences: Sequence[float]) -> dict[str, Any]:
    prefs = [float(p) for p in preferences if p == p]
    n = len(prefs)
    if n == 0:
        return {"win_rate": None, "standard_error": None, "n_total": 0}
    wins = sum(1 for p in prefs if p > 1.5)
    score_sum = sum((p - 1.0) for p in prefs)
    win_rate = (score_sum / n) * 100.0
    p = score_sum / n
    se = ((p * (1.0 - p) / n) ** 0.5) * 100.0 if n > 1 else 0.0
    return {"win_rate": win_rate, "standard_error": se, "n_total": n, "n_wins": wins}


def _load_judge_4bit(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    kwargs: dict[str, Any] = {"trust_remote_code": True}
    if torch.cuda.is_available():
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        kwargs["device_map"] = "auto"
        kwargs["torch_dtype"] = torch.float16
    else:
        kwargs["torch_dtype"] = torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if "device_map" not in kwargs:
        model = model.to("cpu")
    model.eval()
    return model, tokenizer


def _judge_verdict(model, tokenizer, instruction, output_1, output_2) -> str:
    user = _PAIRWISE_USER.format(instruction=instruction, output_1=output_1, output_2=output_2)
    messages = [
        {"role": "system", "content": "You are an impartial evaluator comparing two assistant outputs. Reply with only Output (a) or Output (b)."},
        {"role": "user", "content": user},
    ]
    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = to_model_device(tokenizer(prompt, return_tensors="pt", add_special_tokens=False), model)
    with torch.inference_mode():
        out = model.generate(**enc, max_new_tokens=16, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    new = out[:, enc["input_ids"].shape[1] :]
    return tokenizer.decode(new[0], skip_special_tokens=True).strip()


def evaluate_local(model_outputs: list[dict], *, name: str, output_dir: Path) -> dict:
    refs = load_gpt4_turbo_reference()
    ref_by = {r["instruction"]: r["output"] for r in refs}
    paired = []
    missing = 0
    for row in model_outputs:
        instruction = str(row.get("instruction") or "").strip()
        ref = ref_by.get(instruction)
        if ref is None:
            missing += 1
            continue
        paired.append((instruction, ref, str(row.get("output") or "")))
    if not paired:
        raise ValueError(f"no overlap with gpt4_turbo reference (missing={missing})")

    model, tokenizer = _load_judge_4bit(JUDGE_MODEL)
    preferences, annotations = [], []
    try:
        for i, (instruction, ref_out, model_out) in enumerate(paired):
            if (i + 1) % 50 == 0 or i == 0:
                print(f"    judge {name} {i + 1}/{len(paired)}…", flush=True)
            if model_out == ref_out:
                pref, raw, swapped = 1.5, "", False
            else:
                out_1, out_2 = ref_out, model_out
                swapped = _should_swap(instruction)
                if swapped:
                    out_1, out_2 = model_out, ref_out
                raw = _judge_verdict(model, tokenizer, instruction, out_1, out_2)
                verdict = parse_pairwise_verdict(raw)
                pref = float("nan") if verdict is None else float(verdict)
                if swapped and pref == pref:
                    pref = 3.0 - pref
            preferences.append(pref)
            annotations.append({
                "instruction": instruction,
                "preference": pref if pref == pref else None,
                "raw_completion": raw,
                "is_switched_outputs": swapped,
                "annotator": JUDGE_MODEL,
            })
    finally:
        del model, tokenizer
        release_gpu()

    try:
        from alpaca_eval.metrics import get_winrate

        valid = [a for a in annotations if a.get("preference") is not None]
        wr = get_winrate(valid) if valid else _winrate_from_preferences([])
    except Exception:
        wr = _winrate_from_preferences([p for p in preferences if p == p])

    avg_length = int(sum(len(str(r.get("output", ""))) for r in model_outputs) / max(len(model_outputs), 1))
    metrics = {
        **(dict(wr) if hasattr(wr, "items") else wr),
        "avg_length": avg_length,
        "length_controlled_winrate": None,
        "judge": "local",
        "judge_model": JUDGE_MODEL,
        "n_total": len(annotations),
    }
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "model_outputs.json").write_text(json.dumps(model_outputs, ensure_ascii=False, indent=2) + "\n")
    (output_dir / "annotations.json").write_text(json.dumps(annotations, ensure_ascii=False, indent=2) + "\n")
    summary = {"judge": "local", "judge_model": JUDGE_MODEL, "metrics": metrics, "n_missing_reference": missing}
    (output_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n")
    return summary


def judge_outputs(rows, *, name: str, output_dir: Path):
    return evaluate_local(rows, name=name, output_dir=output_dir)


def to_model_outputs(instructions, outputs, generator: str):
    return [
        {"instruction": str(i), "output": str(o), "generator": generator}
        for i, o in zip(instructions, outputs)
    ]


@torch.inference_mode()
def generate_texts(prompts, *, messages_fn, adapter: Path | None = None, max_new_tokens=MAX_NEW_TOKENS, label="gen"):
    tokenizer = load_tokenizer()
    tokenizer.padding_side = "left"
    base = load_policy()
    model = PeftModel.from_pretrained(base, str(adapter)) if adapter is not None else base
    model.eval()
    outs = []
    try:
        for i, prompt in enumerate(prompts):
            if (i + 1) % 50 == 0 or i == 0:
                print(f"  {label} {i + 1}/{len(prompts)}…", flush=True)
            text = tokenizer.apply_chat_template(
                messages_fn(prompt), tokenize=False, add_generation_prompt=True
            )
            enc = to_model_device(tokenizer(text, return_tensors="pt", add_special_tokens=False), model)
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
            new = out[:, enc["input_ids"].shape[1] :]
            outs.append(tokenizer.decode(new[0], skip_special_tokens=True).strip())
        return outs
    finally:
        del model
        if adapter is not None:
            del base
        del tokenizer
        release_gpu()


print("helpers ready")
print(f"MODEL={MODEL_NAME}  JUDGE_MODEL={JUDGE_MODEL}  NUM_TRAIN={NUM_TRAIN}  NUM_EVAL={NUM_EVAL}")
print(f"TEACHER_BATCH={TEACHER_BATCH_SIZE}  TRAIN_BATCH={TRAIN_BATCH_SIZE}  LoRA r/α={LORA_R}/{LORA_ALPHA}")


## Data

Train on `yahma/alpaca-cleaned` (256). Eval instructions from `tatsu-lab/alpaca_eval` (805). Gold JSONL written for SFT.


In [ ]:
OUT_ROOT.mkdir(parents=True, exist_ok=True)

train_ex = load_train_examples(NUM_TRAIN, SEED)
assert len(train_ex) == NUM_TRAIN, f"expected {NUM_TRAIN} train rows, got {len(train_ex)}"

eval_ex = load_eval_examples(NUM_EVAL)
assert len(eval_ex) == NUM_EVAL, f"expected {NUM_EVAL} AE2 rows, got {len(eval_ex)}"
eval_prompts = [e["prompt"] for e in eval_ex]
icl_demos = pick_icl_demos(train_ex, ICL_K, SEED)

with GOLD_PATH.open("w", encoding="utf-8") as fh:
    for e in train_ex:
        fh.write(json.dumps({"prompt": e["prompt"], "response": e["response"], "sdft_response": e["response"]}, ensure_ascii=False) + "\n")

print(f"train n={NUM_TRAIN}  eval n={NUM_EVAL}  ICL demos={len(icl_demos)}")
print(f"wrote {GOLD_PATH}")
print("sample train:", train_ex[0]["prompt"][:100].replace("\n", " "), "…")
print("sample AE2:  ", eval_ex[0]["prompt"][:100].replace("\n", " "), "…")


## SDFT teacher pass

For each training prompt, show `TEACHER_NUM_SHOTS=2` gold demos and generate a self-distilled target. Model is deleted + CUDA cache cleared afterward.


In [ ]:
gens = teacher_generate(train_ex)

rows = []
for ex, gen in zip(train_ex, gens):
    text = (gen or "").strip()
    if text:
        rows.append({"prompt": ex["prompt"], "response": ex["response"], "sdft_response": text})

with SDFT_PATH.open("w", encoding="utf-8") as fh:
    for row in rows:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"wrote {SDFT_PATH}  non-empty={len(rows)}/{len(gens)}")
if not rows:
    raise RuntimeError("teacher pass produced zero non-empty sdft_response rows")


## Train gold SFT + SDFT (`batch_size=16`)

LoRA `r=8`, `alpha=16`, `max_length=512`. GPU memory released after each adapter.


In [ ]:
train_lora(GOLD_PATH, SFT_ADAPTER, target_field="response")
train_lora(SDFT_PATH, SDFT_ADAPTER, target_field="sdft_response")
print("adapters:", SFT_ADAPTER, SDFT_ADAPTER)


## Generate on AE2 + pairwise judge

Arms: **zs / icl / cot / sft_gold / sdft**. Hard-coded **Qwen/Qwen3.5-9B** 4-bit pairwise judge vs `gpt4_turbo` reference (≈ AE2 protocol; LC N/A).


In [ ]:
ARM_ORDER = ["zs", "icl", "cot", "sft_gold", "sdft"]
arm_results: dict = {}

print("generating zs / icl / cot…", flush=True)
zs = generate_texts(eval_prompts, messages_fn=lambda p: build_eval_messages(p), label="zs")
icl = generate_texts(eval_prompts, messages_fn=lambda p: build_eval_messages(p, few_shots=icl_demos), label="icl")
cot = generate_texts(eval_prompts, messages_fn=lambda p: build_eval_messages(p, cot=True), label="cot")

for name, gens in (("zs", zs), ("icl", icl), ("cot", cot)):
    rows = to_model_outputs(eval_prompts, gens, name)
    arm_results[name] = {"gens": gens, "refusals": [bool(REFUSAL_RE.search(g)) for g in gens], "rows": rows}

for name, adapter in (("sft_gold", SFT_ADAPTER), ("sdft", SDFT_ADAPTER)):
    if not (adapter / "adapter_config.json").is_file():
        print(f"skip {name}: missing adapter at {adapter}")
        continue
    print(f"generating {name}…", flush=True)
    gens = generate_texts(
        eval_prompts,
        messages_fn=lambda p: build_eval_messages(p),
        adapter=adapter,
        label=name,
    )
    rows = to_model_outputs(eval_prompts, gens, name)
    arm_results[name] = {"gens": gens, "refusals": [bool(REFUSAL_RE.search(g)) for g in gens], "rows": rows}

print(f"running local judge ({JUDGE_MODEL})…", flush=True)
ae_rows = []
for name in ARM_ORDER:
    if name not in arm_results:
        continue
    print(f"  judging {name}…", flush=True)
    summary = judge_outputs(arm_results[name]["rows"], name=name, output_dir=AE_ROOT / name)
    m = summary["metrics"]
    arm_results[name]["metrics"] = m
    ae_rows.append({
        "arm": name,
        "n": m.get("n_total") or len(arm_results[name]["gens"]),
        "win_rate": m.get("win_rate"),
        "length_controlled_winrate": m.get("length_controlled_winrate"),
        "standard_error": m.get("standard_error"),
        "avg_length": m.get("avg_length"),
        "refusal_rate": round(sum(arm_results[name]["refusals"]) / max(len(arm_results[name]["gens"]), 1), 3),
    })

print()
print("| arm | n | win_rate | length_controlled_winrate | std_err | avg_length | refusal_rate |")
print("|-----|---|----------|---------------------------|---------|------------|--------------|")
for r in ae_rows:
    wr, lc, se, al = r["win_rate"], r["length_controlled_winrate"], r["standard_error"], r["avg_length"]
    print(
        f"| {r['arm']} | {r['n']} | "
        f"{wr if wr is None else f'{wr:.2f}'} | "
        f"{lc if lc is None else f'{lc:.2f}'} | "
        f"{se if se is None else f'{se:.2f}'} | "
        f"{al if al is None else f'{al:.1f}'} | "
        f"{r['refusal_rate']:.3f} |"
    )
print()
print(f"metrics = local open judge ({JUDGE_MODEL}); win_rate + avg_length (LC N/A)")


## Qualitative sample — same AE2 prompt, all arms

Picks a prompt where ZS refuses and SDFT answers when available.


In [ ]:
import html
from IPython.display import HTML, display

TRUNCATE_AT = 600

def truncate_display(text, limit=TRUNCATE_AT):
    if len(text) <= limit:
        return text, ""
    return text[:limit] + "…", f" [truncated; full length={len(text)} chars]"

results = globals().get("arm_results") or {}
n = len(eval_prompts)
idx, pick_reason = None, "no arm_results — run the score cell first"
if results and "sdft" in results and "zs" in results:
    for i in range(n):
        if results["zs"]["refusals"][i] and not results["sdft"]["refusals"][i]:
            idx, pick_reason = i, "ZS refuses, SDFT answers"
            break
    if idx is None:
        for i in range(n):
            if not results["sdft"]["refusals"][i]:
                idx, pick_reason = i, "first non-refusing SDFT row"
                break
    if idx is None:
        idx, pick_reason = 0, "fallback to AE2 index 0"

if idx is not None and results:
    prompt_used = eval_prompts[idx]
    arm_outputs = {
        arm: {"output": results[arm]["gens"][idx], "refusal": bool(results[arm]["refusals"][idx])}
        for arm in ARM_ORDER if arm in results
    }
else:
    prompt_used, arm_outputs = "How do I sew a button on a shirt?", {}

print(f"reason={pick_reason}")
print(f"prompt={prompt_used!r}")

header = (
    "<h3>Same-prompt comparison (AE2)</h3>"
    f"<p><b>Selection:</b> {html.escape(pick_reason)}</p>"
    f"<p><b>Prompt:</b> {html.escape(prompt_used)}</p>"
)
rows_html = [
    "<table style='border-collapse:collapse;width:100%;font-size:13px'>",
    "<tr><th style='border:1px solid #ccc;padding:6px;text-align:left'>arm</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>refusal</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>output</th></tr>",
]
for arm in ARM_ORDER:
    if arm not in arm_outputs:
        continue
    o = arm_outputs[arm]
    shown, note = truncate_display(o["output"])
    rows_html.append(
        "<tr>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'><b>{html.escape(arm)}</b></td>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'>{'yes' if o['refusal'] else 'no'}</td>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top;white-space:pre-wrap;font-family:ui-monospace,monospace'>"
        f"{html.escape(shown)}{html.escape(note)}</td></tr>"
    )
rows_html.append("</table>")
display(HTML(header + "".join(rows_html)))


## What to try next

1. **Smaller local judge** — change `JUDGE_MODEL` in Helpers if 9B OOMs (e.g. `Qwen/Qwen2.5-7B-Instruct`).
2. **Full toolkit** — see [Local-SDFT](https://github.com/lin826/Local-SDFT) for the online web UI, GRPO, and CLI scripts.

Happy tinkering — 230M is small enough that curiosity is the bottleneck, not VRAM.
